In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load raw data

In [ ]:
df = pd.read_csv('../../data/processed/processed_data.csv')

In [ ]:
df[df.duplicated()]

### Check for outliers
Do not remove during data preparation. Outliers are crutial for detecting anomalies such as fraud transactions.

In [ ]:
cat_cols, cont_cols, binary_cols = [], [], []

for col in df.columns:
    if df[col].dtype == "str":
        cat_cols.append(col)
    else:
        if df[col].nunique() > 2:
            cont_cols.append(col)
        else:
            binary_cols.append(col)


print("Categorical Columns:", cat_cols)
print("Continuous Columns:", cont_cols)
print("Binary Columns:", binary_cols)

In [ ]:
feature_dict = {"global" : [], "customer": [], "merchant": [], "other": []}

for col in cont_cols:
    if col.startswith("global_"):
        feature_dict["global"].append(col)
    elif col.startswith("customer"):
        feature_dict["customer"].append(col)
    elif col.startswith("merchant"):
        feature_dict["merchant"].append(col)
    else:
        feature_dict["other"].append(col)

print("Global Features:", feature_dict["global"])
print("Customer Features:", feature_dict["customer"])
print("Merchant Features:", feature_dict["merchant"])
print("Other Features:", feature_dict["other"])

### Check class distribution
If classes are imbalanced, avoid using Accuracy metric. Use F1 Score, Percision and Recall instead to measure model preformance.

In [ ]:
# count values of classifier column
fraud_counts = df['fraud'].map({0: "No", 1: "Yes"}).value_counts()

bin_dict = {"age": [], "gender": []}
for col in binary_cols:
    if col.startswith("age"):
        bin_dict["age"].append(col)
    elif col.startswith("gender"):
        bin_dict["gender"].append(col)

age_counts = df[bin_dict["age"]].sum()
gender_counts = df[bin_dict["gender"]].sum()

print("Fraud Counts:", fraud_counts)
print("\nAge Counts:\n", age_counts)
print("\nGender Counts:\n", gender_counts)

In [ ]:
def pie_with_arrows(ax, data, title):
    wedges, _ = ax.pie(data, startangle=90)
    ax.set_title(title)

    for i, wedge in enumerate(wedges):
        angle = (wedge.theta2 + wedge.theta1) / 2
        x = np.cos(np.deg2rad(angle))
        y = np.sin(np.deg2rad(angle))

        label = f"{data.index[i]}: {data.iloc[i] / data.sum() * 100:.1f}%"

        ax.annotate(
            label,
            xy=(x, y),
            xytext=(1.2 * np.sign(x), 1.2 * y),
            arrowprops=dict(arrowstyle="-"),
            ha="left" if x > 0 else "right"
        )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 10))

pie_with_arrows(axes[0], fraud_counts, "Fraud Distribution")
pie_with_arrows(axes[1], age_counts, "Age Distribution")
pie_with_arrows(axes[2], gender_counts, "Gender Distribution")

plt.tight_layout()
plt.show()

### Variance Calculation
Looks at the number of unique values in a column if its just 1 drop it


In [ ]:
cols_to_drop = []
for col in df.columns:
   unum = df[col].nunique()

   print(f"Unique numbers of {col}s:", unum)

   if unum == 1:
      cols_to_drop.append(col)

print("\nDropping columns due to constant values:", cols_to_drop)

for col in cols_to_drop:
   cat_cols.remove(col)

df.drop(columns=cols_to_drop, inplace=True)

### Compare competing features

In [ ]:
mean = df["customer_amount"].mean()
median = df["customer_amount"].median()
q99 = df["customer_amount"].quantile(0.99)

fraud_values = df["customer_amount"][df["fraud"] == 1]
normal_values = df["customer_amount"][df["fraud"] == 0]

fig, axes = plt.subplots(3, 1,figsize=(40, 18))

_, bin_edges, _ = axes[0].hist(df["customer_amount"], bins=50, range=(df["customer_amount"].min(),q99))

axes[0].set_title("Customer Amount Distribution")
axes[0].set_xticks(bin_edges)
axes[0].set_xlabel("Customer Amount")
axes[0].set_ylabel("Frequency")

axes[0].axvline(mean, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean:.2f}')
axes[0].axvline(median, color='green', linestyle='solid', linewidth=2, label=f'Median: {median:.2f}')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.7)

# Density captures the probability distribution, probability = max_bin_width * height(density)
axes[1].hist(normal_values, bins=50, alpha=0.6, label='Normal', density=True, range=(normal_values.min(), q99))
axes[1].hist(fraud_values, bins=50, label='Fraud', color="orange", density=True, range=(fraud_values.min(), q99))

axes[1].set_title("Customer Amount Distribution by Fraud Status")
axes[1].set_xticks(bin_edges)
axes[1].set_xlabel("Customer Amount")
axes[1].set_ylabel("Probability Density")

axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.7)

axes[2].boxplot(
    [
        normal_values,
        fraud_values
    ],
    vert=False,
    showfliers=False,
    labels=["Normal", "Fraud"]
)
axes[2].set_title("Customer Amount Box Plot")


### Correlation Calculation and Visualisation
Sees if any two features have a linear relationship.

In [ ]:
cor_cof= df[cont_cols + ["fraud"]].corr()
mask = np.triu(np.ones_like(cor_cof, dtype=bool), k=1) # Correlation is symmetric, so we only need to show one triangle of the matrix

plt.figure(figsize=(20, 16))
sns.heatmap(cor_cof, mask=mask, annot=True, vmin=-1, vmax=1, cbar_kws={"aspect": 40}, cmap="viridis", fmt=".2f", xticklabels=cor_cof.columns, yticklabels=cor_cof.columns, square=True, annot_kws={"size": 10})
plt.title("Correlation between features")
plt.tight_layout()
plt.show()


In [ ]:
target_corr = cor_cof['fraud']
target_corr_cont = target_corr[cont_cols] # Ignores binarry features as they are not useful for scatter plots

top2_vals = target_corr_cont.abs().sort_values(ascending=False)[:2] #Find the top 2 features most correlated with fraud
print(top2_vals)

top2_features = top2_vals.index.to_list()

In [ ]:
f1, f2 = top2_features

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=df[f1],
    y=df[f2],
    hue=df['fraud'],
    alpha=0.7,
)

plt.show()

### Before data preparation

*Feature selection* - Are there features with no significance or zero variance?

*Feature scaling* - Are there numerical features with vastly different ranges?

*Data leakage* - Are there features that would not be available at the time of prediction but perfectly predict the output?

*Stratification* - Will the percentage of each class be roughly the same across data splits?

*Order* - Is your data time based? Can you shuffle it?

*Hidden bias* - Does data accurately represent the real world?
